
# Antibody Classification from Prepared Embeddings (Start Here)

This tutorial **starts after you have already computed sequence embeddings** for each sample. 
In other words, you should arrive here with a Python object like `data: List[Dict]` on disk. 
We will **not** show how to generate embeddings here; instead, we will train a 1D CNN classifier 
from the prepared embeddings.

---



## 0. Prerequisites

Please prepare a serialized list named `data` (saved as `data.npy` via NumPy), 
where each element is a dictionary describing **one sample** (e.g., one participant). For each sample, 
its fields should look like:

- `data[0].keys()` → `['seqs', 'cdr3', 'cropped_seqs', 'embs', 'n_counts', 'label']`

Field definitions:
- **`seqs`** (`List[str]`): All **original** amino-acid sequences belonging to this sample.
- **`cdr3`** (`List[str]`): The **CDR3 subsequence** for each corresponding sequence in `seqs`.
- **`cropped_seqs`** (`List[str]`): The **cropped** sequences actually fed into the encoder.
  Each cropped sequence **contains** the CDR3 region (cropped around CDR3 using a normal window).
- **`embs`** (`torch.Tensor` of shape `[N, 768]` by default): Concatenation of per-sequence embeddings 
  produced by the pretrained antibody model for each cropped sequence in `cropped_seqs`.
- **`n_counts`** (`List[int]`): Occurrence count for each sequence (e.g., `DUPCOUNT` from your source table).
- **`label`** (`int`): The sample label (e.g., 0/1; in your RA/AD study, `1` may indicate Alzheimer's Disease).


## 1. Environment and Imports

Make sure you are in a Huggingface official Docker container: <br>
`docker.io/huggingface/transformers-pytorch-gpu:4.29.1`  <a href="https://hub.docker.com/r/huggingface/transformers-pytorch-gpu">link</a>

Then import the necessary modules:


In [ ]:
!pip install monai==1.0.1 info-nce-pytorch==0.1.4 gdown

In [ ]:
import os
from typing import Any, Dict, List
import gdown

import numpy as np
import torch

from classification.model import CNN1DClassifier
from classification.data_utils import create_dataloaders
from classification.train import train_model, plot_training_history

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 2. Configure Path to Prepared Embeddings

Point `PREPARED_EMB_FILE` to your serialized `data` object. In the examples below, we will demonstrate 4 samples from two public datasets. Please refer to their official pages:
- 1st: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE242738
- 2nd: https://zenodo.org/records/3585046

In [ ]:
# Download the samples
dir_name = 'samples'
if not os.path.exists(dir_name):
    os.mkdir(dir_name)
url = 'https://drive.google.com/file/d/1a3tGR32BPMZlwVzv2q4GnA2HRl0lsCS5/view?usp=sharing'
output_file = dir_name+'/data.npy'
gdown.download(url=url, output=output_file, fuzzy=True)

In [ ]:
# Set your path to the prepared `data` list
PREPARED_EMB_FILE = "./samples/data.npy" # or your own npy path 

def load_prepared_data(path: str) -> List[Dict[str, Any]]:
    path_lower = path.lower()
    loaded = np.load(path, allow_pickle=True)
    samples = loaded.tolist()

    # Normalize: ensure 'embs' are numpy float32 arrays of shape (N, D)
    norm_samples: List[Dict[str, Any]] = []
    for s in samples:
        entry = dict(s)  # shallow copy
        embs = entry.get("embs", None)
        if embs is None:
            raise ValueError("Each sample must contain an 'embs' field.")

        if isinstance(embs, torch.Tensor):
            embs = embs.detach().cpu().numpy()
        embs = np.asarray(embs, dtype=np.float32)
        entry["embs"] = embs

        # Ensure label is float (binary 0.0/1.0 expected)
        if "label" in entry:
            entry["label"] = float(entry["label"])
        else:
            raise ValueError("Each sample must contain a 'label' field.")

        norm_samples.append(entry)
    return norm_samples

samples = load_prepared_data(PREPARED_EMB_FILE)
print(f"Loaded {len(samples)} samples.")

## 3. Train/Validation Split and Dataloaders
In this step, we prepare our data for the training loop. This involves two key actions:

1.  **Splitting the Data:** We manually divide our `samples` list into a training set and a validation set. This allows us to monitor the model's performance on unseen data during training.
2.  **Initializing DataLoaders:** We convert these lists into `DataLoader` objects. We use a batch size of `1` here, which is often safer for handling sequences of varying or very long lengths without running into memory issues. We also allow the loader to infer the `max_length` dynamically from the data.

In [ ]:
# Split into train / valid
train_samples, valid_samples = samples[:2], samples[2:]
print(f"Train samples: {len(train_samples)}, valid samples: {len(valid_samples)}")

# Create DataLoaders
train_loader, valid_loader, max_len = create_dataloaders(
    train_samples=train_samples,
    valid_samples=valid_samples,
    batch_size=1,     # batch size 1 handles very large samples robustly
    max_length=None,  # infer from data
    num_workers=0,
)


## 4. Initialize and Train the CNN Classifier


In [ ]:
model = CNN1DClassifier(in_channels=train_loader.dataset.embedding_dim, num_classes=1)
history = train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    num_epochs=100,   
    lr=1e-4,    
    enable_contrastive=False,
    checkpoint_dir='./',
    device=device,
)

In [ ]:
plot_training_history(history)